# CLASSIFIER

I need to freeze transformer weights and train only classifier head.

sources: 
https://perso.esiee.fr/~chierchg/deep-learning/quickref/quickref-transfer.html
https://perso.esiee.fr/~chierchg/deep-learning/tutorials/transfer/transfer-3.html
https://stackoverflow.com/questions/51433378/what-does-model-train-do-in-pytorch



on the transformer: require_grad = False to freeze the weights, eval 
We can use the parameters() iterator to access the parameters of a nn.Module and set the requires_grad attribute accordingly.


In [1]:
from protgen.models import DecoderOnlyTransformer
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn


In [2]:
# Constants for the dataset and model
FAMILIES = ("AcyP", "HPr", "HU")
AA = "ACDEFGHIKLMNPQRSTVWY"
special_tokens = ["<PAD>", "<BOS>", "<EOS>"]
total_tokens = list(special_tokens) + list(AA)
IGNORE_INDEX = -1  # what F.cross_entropy(ignore_index=...) must be given for the generative task




train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Indexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [ ]:
from torch import nn
# template from https://www.learnpytorch.io/02_pytorch_classification/#8-putting-things-together-by-building-a-multi-class-pytorch-model
# Build model
class ClassifierHead(nn.Module):
    
    # input from transformer: (batch_size, seq_len, embed_dim) = hidden states
    # it takes onlz the last dimension: embed_dim
    # output: (batch_size, seq_len, num_classes) -> class logits per sample
    
    def __init__(self, input_features, output_features, num_layers, hidden_units=128):
        
        """Initializes all required hyperparameters for a multi-class classification model.

        Args:
            input_features (int): Number of input features to the model.
            out_features (int): Number of output features of the model
              (how many classes there are).
            hidden_units (int): Number of hidden units between layers, default 128.
        """# 128 hidden units is similar to embedding dimension and seq length
        super().__init__()
        
        layers = []
        
        for i in range(num_layers):
            layers.append(nn.Linear(in_features=input_features if i == 0 else hidden_units, out_features=hidden_units))
            layers.append(nn.GELU())
        layers.append(nn.Linear(in_features=input_features if num_layers == 0 else hidden_units, out_features=output_features))
        
        self.linear_layer_stack = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.linear_layer_stack(x)

In [26]:
# test with data and transformer output


example_protein_seq_data = torch.stack([train_data[0][0], train_data[1][0]])
example_protein_seq_labels = torch.stack([train_data[0][2], train_data[1][2]])

print(example_protein_seq_data.shape) # batch_size, seq_len
print(example_protein_seq_labels)

embed_dim = 128
num_heads = 4
num_layers = 2
max_seq_length = 129
pad_idx = train_data.vocab["<PAD>"]  # 0


babys_first_transformer = DecoderOnlyTransformer(vocab_size=train_data.vocab_size,
                                                 embedding_dim=embed_dim,
                                                 num_heads=num_heads,
                                                 ff_dim=embed_dim*4,
                                                 num_layers=2,
                                                 max_seq_length=max_seq_length,
                                                 pad_idx=pad_idx,
                                                 dropout=0.1)


hidden_states = babys_first_transformer(example_protein_seq_data, return_hidden_states=True)

hidden_states.shape # batch_size, seq_len, embed_dim


# now classifier head
classifier_head = ClassifierHead(input_features=embed_dim, output_features=len(FAMILIES))
output = classifier_head(hidden_states) 
output

# output looks right for a classifier that knows nothing
# with the input of hidden states from transformer
# who also knows nothing
# :)


torch.Size([2, 129])
tensor([1, 2])


tensor([[[ 0.3608, -0.2189,  0.2436],
         [ 0.1925, -0.1482,  0.1093],
         [-0.0840,  0.2381,  0.1065],
         [-0.0734,  0.1539,  0.0685],
         [-0.0313, -0.0753,  0.1379],
         [ 0.1944,  0.2166,  0.1418],
         [-0.3306,  0.2657,  0.0991],
         [ 0.2408, -0.1424,  0.1777],
         [ 0.2447,  0.4135,  0.2767],
         [ 0.0924,  0.1182,  0.1095],
         [-0.0107,  0.2267,  0.1047],
         [ 0.0972,  0.0605,  0.2023],
         [ 0.0566,  0.0986,  0.1191],
         [-0.1892,  0.3502,  0.5254],
         [-0.2458,  0.1124,  0.1519],
         [-0.2577, -0.0943,  0.1901],
         [-0.0071,  0.3800,  0.2730],
         [ 0.1845, -0.0120,  0.5330],
         [-0.2834, -0.1014,  0.2302],
         [-0.2672,  0.2570, -0.3044],
         [-0.1618,  0.1164,  0.1198],
         [ 0.0221, -0.2029,  0.0565],
         [ 0.2778,  0.4216, -0.1352],
         [ 0.0902,  0.0366,  0.0786],
         [ 0.0827,  0.1826,  0.0774],
         [-0.0132,  0.1026,  0.0481],
         [ 0

https://perso.esiee.fr/~chierchg/deep-learning/tutorials/transfer/transfer-3.html: 

To freeze a layer, we set the requires_grad attribute to False for all parameters in that layer. This prevents the optimizer from updating the weights of the layer during training. We can use the parameters() iterator to access the parameters of a nn.Module and set the requires_grad attribute accordingly. 

        for params in architecture.features.parameters():
            params.requires_grad = False

In [ ]:
class ClassifierGuidedLM(nn.Module): 
    
    def __init__(self, transformer, classifier_head):
        
        
        super().__init__()
        
        # change transformer settings to freeze it 
        transformer.eval()  # set transformer to evaluation mode
        for param in transformer.parameters():
            param.requires_grad = False  # freeze transformer weights
            
        self.transformer = transformer
        self.classifier_head = classifier_head
        
    # now i freeze transformer weights -> put it to eval at all times
    # need to override / disable training mode for transformer completely ?

    def forward(self, x):
        self.transformer.eval()  # make sure transformer is in evaluation mode again
        hidden_states = self.transformer(x, return_hidden_states=True)
        output = self.classifier_head(hidden_states)
        return output
    
 

In [ ]:
### FINAL TESTING 

final_model = ClassifierGuidedLM(transformer=babys_first_transformer, 
                                          classifier_head=classifier_head)

output = final_model(example_protein_seq_data)
print(output.shape) # batch_size, seq_len, num_classes

# train() and eval() should have same output
# because no parameters should be changed
# train
final_model.train()
print("before forward call:", final_model.transformer.training)
output_train1 = final_model(example_protein_seq_data)
output_train2 = final_model(example_protein_seq_data)
print("after forward call:", final_model.transformer.training)
print("equal outputs before and after training:", torch.equal(output_train1, output_train2)) 

final_model.eval()
output_eval = final_model(example_protein_seq_data)
print("after eval call:", final_model.transformer.training)
print("equal outputs in training and eval:", torch.equal(output_train1, output_eval))

# check trainable parameters
trainable = [n for n, p in final_model.named_parameters() if p.requires_grad]
print(trainable)
# only classifier head :))

torch.Size([2, 129, 3])
before forward call: True
after forward call: False
equal outputs before and after training: True
after eval call: False
equal outputs in training and eval: True
['classifier_head.linear_layer_stack.0.weight', 'classifier_head.linear_layer_stack.0.bias', 'classifier_head.linear_layer_stack.2.weight', 'classifier_head.linear_layer_stack.2.bias']
